# Bài thực hành 1: Xây dựng giao diện người dùng cơ bản cho agent

Trong bài học này, bạn sẽ học cách kết nối một LangChain agent với giao diện chat React thông qua **CopilotKit** và giao thức **AG-UI**. Đây là một mô hình chuẩn mực có thể được sử dụng để kết nối bất kỳ backend agent nào với bất kỳ frontend nào.

**Mục tiêu bài học**:

- **Khởi chạy một LangChain Agent:** Xây dựng một backend tương thích với AG-UI bằng FastAPI.
- **Thiết lập CopilotKit:** Kết nối giao diện frontend (React) với agent thông qua `CopilotRuntime`.
- **Chuyển đổi backend linh hoạt:** Thay đổi qua lại giữa LangChain/OpenAI và Google ADK/Gemini mà không cần sửa đổi mã nguồn của giao diện (UI).

## 1. Chuẩn bị môi trường

Trước khi bắt đầu, chúng ta cần cài đặt các thư viện cần thiết và cấu hình API Key.

Lưu ý: Trong môi trường thực hành này (Jupyter Notebook), chúng ta sử dụng lệnh %%writefile để lưu code trực tiếp vào thư mục dự án frontend. Ứng dụng sẽ tự động cập nhật (hot-reload) để bạn có thể xem các thay đổi theo thời gian thực.

Khởi tạo môi trường bằng các đoạn mã sau:

In [1]:
# Bỏ qua các cảnh báo không cần thiết
import warnings
warnings.filterwarnings("ignore")

# Cài đặt các thư viện frontend (npm install)
from helper import install_frontend
install_frontend()

# Tải API Keys (OpenAI, Gemini, v.v.)
from helper import load_api_keys
load_api_keys()

Installing frontend dependencies ...
npm warn deprecated hast@1.0.0: Renamed to rehype
npm warn deprecated lodash.get@4.4.2: This package is deprecated. Use the optional chaining (?.) operator instead.
npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead

added 970 packages, and audited 971 packages in 11s

249 packages are looking for funding
  run `npm fund` for details

32 vulnerabilities (7 low, 19 moderate, 6 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm warn allow-scripts 3 packages have install scripts not yet covered by allowScripts:
npm warn allow-scripts   @scarf/scarf@1.4.0 (postinstall: node ./report.js)
npm warn allow-scripts   esbuild@0.27.7 (postinstall: node install.js)
npm warn allow-scripts   esbuild@0.25.12 (postinstall: node install.js)
npm warn allow-scripts
npm warn allow-scripts 

**Lưu ý khi chạy trên máy cá nhân:** Bạn sẽ cần tự cung cấp API key của mình:
- **OpenAI:** [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **Google AI:** [aistudio.google.com/apikey](https://aistudio.google.com/app/apikey)

## 2. Xây dựng agent backend

### 2.1. Khởi động server

CopilotKit kết nối với agent của bạn thông qua một endpoint HTTP tương thích với AG-UI. Dưới đây, chúng ta sẽ khởi chạy một server FastAPI và gắn một LangGraphAGUIAgent vào đó.

In [2]:
from fastapi import FastAPI

# Import các thư viện phụ thuộc của CopilotKit và AG-UI
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent
from helper import start_server

# Khởi tạo ứng dụng FastAPI và thiết lập endpoint AG-UI
app = FastAPI()
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="lesson2_agent",
    description="Lesson 2 chart agent",
    graph=graph,
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Khởi động server ở port 8002
start_server(app, port=8002)

✓ Server running at http://localhost:8002


### 2.2. Định nghĩa agent

Tiếp theo, chúng ta tạo một LangChain agent sử dụng model của OpenAI, kết hợp với bộ nhớ (memory checkpointer) và CopilotKit middleware.

**Tại sao cần CopilotKitMiddleware?** Middleware này chính là cầu nối cho phép model khám phá và gọi các công cụ được định nghĩa ở phía frontend (bạn sẽ học ở bài 3). Nếu không có middleware này, agent chỉ có thể nhìn thấy các công cụ được định nghĩa ở backend.


In [3]:
from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Tạo một LangChain agent
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()], # Quan trọng để kết nối với Frontend
    checkpointer=MemorySaver(),
    system_prompt=("You are a helpful assistant"),
)

# Cập nhật graph cho agent (Tính năng hot-reload, không cần khởi động lại server)
agent.graph = graph
print("✓ Cập nhật agent graph thành công!")

✓ Cập nhật agent graph thành công!


## 3. Bắt đầu với CopilotKit (Frontend)

CopilotKit cung cấp cả giao diện UI thuần (không có style mặc định) và các component React được thiết kế sẵn cho UI của agent. Trong bài này, chúng ta dùng 3 thành phần chính:

- **CopilotRuntime:** Cây cầu bảo mật kết nối frontend với bất kỳ agent backend nào.
- **CopilotKit:** React Provider dùng để cấu hình kết nối runtime cho ứng dụng của bạn.
- **CopilotChat:** Giao diện chat hoàn chỉnh, tích hợp sẵn mọi tính năng và có thể tùy biến.

Đầu tiên, khởi động frontend dev server:

In [ ]:
from helper import start_frontend, display_app
start_frontend(port=3002)
display_app(port=3002) # Mở ứng dụng trong trình duyệt để xem trực tiếp

### 3.1. Thiết lập `CopilotRuntime`

Chúng ta sẽ tạo file `server.ts` ở phía frontend để đăng ký LangChain agent vừa tạo (chạy ở cổng 8002) dưới tên là `default` agent.

In [5]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


Lưu ý: Chúng ta sử dụng đường dẫn /v2 vì nó cung cấp các hook và component mới nhất.

### 3.2. Bao bọc ứng dụng bằng `CopilotKit` Provider

Bước tiếp theo là bọc toàn bộ ứng dụng React bằng component `<CopilotKit>` và trỏ nó tới `runtimeUrl` mà chúng ta vừa tạo.

In [6]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

Overwriting frontend/src/main.tsx


### 3.3. Thiết lập component `CopilotChat`

Cuối cùng, thêm giao diện chat vào ứng dụng chính và chỉ định `agentId` là `"default"` để kết nối với LangChain agent.

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Bây giờ, ứng dụng của bạn đã có một giao diện chat hoàn chỉnh có thể giao tiếp với LangChain agent! Hãy thử gõ *"Please write me a poem!"* hoặc *"Hello chat!"* để kiểm tra.

## 4. Mở rộng: Kết nối với Google ADK (Gemini)

Trong phần này, chúng ta sẽ thêm một agent backend thứ hai sử dụng **Google ADK và mô hình Gemini**, sau đó chuyển đổi qua lại mà **không cần thay đổi bất kỳ code UI nào.**

### 4.1. Thiết lập ADK agent

Khởi chạy một Google ADK agent trên port `8009`:

In [8]:
from fastapi import FastAPI
from ag_ui_adk import ADKAgent, add_adk_fastapi_endpoint
from google.adk.agents import LlmAgent
from helper import start_server

# Khởi tạo mô hình Gemini
gemini_agent = LlmAgent(
    name="assistant",
    model="gemini-2.5-flash",
    instruction="Be helpful and fun!",
)

# Bọc trong ADK Agent
adk_agent = ADKAgent(
    adk_agent=gemini_agent,
    app_name="demo_app",
    user_id="demo_user",
    session_timeout_seconds=3600,
    use_in_memory_services=True,
)

app_adk = FastAPI()
add_adk_fastapi_endpoint(app_adk, adk_agent, path="/")

# Khởi động server phụ ở port 8009
start_server(app_adk, port=8009)

✓ Server running at http://localhost:8009


### 4.2. Cập nhật `CopilotRuntime` để đăng ký agent mới

Chỉnh sửa lại file `server.ts` để `CopilotRuntime` nhận diện cả 2 agents:

In [9]:
%%writefile frontend/server.ts

import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { HttpAgent } from "@ag-ui/client";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";
import { serve } from "@hono/node-server";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const adkAgent = new HttpAgent({
  url: process.env.ADK_AGENT_URL || "http://localhost:8009",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
    gemini: adkAgent,
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


### 4.3. Đổi sang dùng Gemini trên giao diện

Giờ đây, bạn chỉ cần thay đổi biến `agentId` trong file `App.tsx` từ `"default"` sang `"gemini"`:

In [10]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

export const agentId = "gemini";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


Hãy thử hỏi ứng dụng: *"What model are you powered by?"*. Bạn sẽ thấy phản hồi lúc này đến từ Gemini!

## 5. Tìm hiểu lý thuyết: AG-UI là gì?

**AG-UI (Agent-User Interaction)** là một giao thức mã nguồn mở, hoạt động dựa trên sự kiện nhằm kết nối các backend agent với giao diện frontend.

Nó chuẩn hóa cách các tin nhắn chat, lệnh gọi công cụ, cập nhật trạng thái, và luồng token được truyền tải qua HTTP.

Tại sao giao thức này lại quan trọng?

- **Tính tương thích:** CopilotKit có thể giao tiếp với **bất kỳ** backend nào tuân thủ chuẩn AG-UI.
- **Tính linh hoạt:** Như bạn vừa thấy, ta có thể chuyển từ LangChain/OpenAI sang ADK/Gemini chỉ bằng một dòng cấu hình `agentId` - hoàn toàn không phải viết lại code frontend.
- **Tính đồng nhất:** Hành vi streaming văn bản và gọi công cụ luôn nhất quán bất kể bạn dùng framework backend nào.

<img src="images/protocols.png" alt="AG-UI protocol diagram" style="max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## 6. Tổng kết

Trong bài học này, bạn đã học được:

- Cách chạy một LangChain agent sau một endpoint AG-UI và kết nối nó với CopilotKit.
- Kỹ thuật hot-reload graph của agent trong quá trình phát triển để tiết kiệm thời gian.
- Sức mạnh của giao thức chuẩn: Sử dụng cùng một frontend nhưng dễ dàng chuyển đổi qua lại giữa các backend (LangChain/OpenAI và ADK/Gemini).

**Bước tiếp theo:**

Trong bài tiếp theo, chúng ta sẽ đi sâu vào GenUI có kiểm soát, bao gồm:

- Đăng ký các công cụ frontend có định dạng kiểu dữ liệu bằng hook `useComponent()`.
- Hiển thị kết quả của các công cụ này trực tiếp trong luồng chat.
- Đảm bảo hành vi của giao diện (do model điều khiển) luôn an toàn và có thể dự đoán được.
